In [ ]:
# Start Snowpark session
from snowflake.snowpark.session import Session

# Use the Snowflake context variables for the notebook (usually preconfigured)
session = Session.builder.getOrCreate()

# Verify the connection by running a simple query
session.sql("SELECT CURRENT_USER(), CURRENT_REGION(), CURRENT_ROLE()").show()


I'll be using the Tasty Bytes sample data as it is free.

In [ ]:
df = session.table("tasty_bytes_sample_data.raw_pos.menu")
df.limit(5).show()

In Snowpark for Python, `df.limit(5).show()` is used instead of `df.head()` because it runs the query directly in Snowflake and only displays a limited preview of the data without transferring it to your local environment. This approach is more efficient and cost-effective, especially when working with large or petabyte-scale datasets, as it avoids unnecessary data movement and memory usage. In contrast, `df.head()` pulls the data into a local Pandas DataFrame, which can be slower and more resource-intensive.

## Exploring with Snowpark DataFrame API

In [ ]:
df.schema

In [ ]:
df_filtered = df.filter(df["truck_brand_name"] == "Freezing Point")
df_filtered.select("menu_item_name", "sale_price_usd").show()

In [ ]:
df.group_by("truck_brand_name").agg({"sale_price_usd": "avg"}).show()

Work with JSON/Semi-Structured Data

In [ ]:
from snowflake.snowpark.functions import col

df_flat = df.select(
    col("menu_item_name"),
    col("menu_item_health_metrics_obj")["menu_item_health_metrics"]["ingredients"].alias("ingredients")
)
df_flat.show()

In [ ]:
session.sql("CREATE SCHEMA IF NOT EXISTS My_schema").collect()

In [ ]:
df_filtered.write.mode("overwrite").save_as_table("My_schema.filtered_menu")

In [ ]:
df_filtered.to_pandas().to_csv("filtered_menu.csv", index=False)